# Dev Eval: Gemma 4 Developer Agent

Runs a fixed 60-task dev set from the public `tasks.jsonl` against **your** `submission.zip`, and writes the files that `eval/triage.py` and `eval/trace_stats.py` read.

**Before running**
1. Locally: `python scripts/package_submission.py` → `dist/submission.zip` (prints a fingerprint).
2. Upload `dist/submission.zip` as a Kaggle Dataset (new version for every change). Kaggle may auto-extract it; both forms work.
3. Attach to this notebook: the competition data, the `google/gemma-4` model (`gemma-4-31b-it-qat-w4a16-ct`), the wheelhouse dataset, and your submission dataset. Accelerator **GPU L4x4**, Internet **off**.
4. Edit the **Run config** cell (`RUN_NAME` at least).

**Run** with *Save Version → Save & Run All (Commit)* so it survives closing the browser. 60 tasks take roughly 6–8 h (L4x4 quota is charged 2×). Check that the printed submission fingerprint matches step 1.

**After** — download `results_<RUN_NAME>.zip` from the Output tab, unzip into `exp_res/results_<RUN_NAME>/`, then run the triage commands printed by the last cell.

## 0. Run Config

In [ ]:
import time

NOTEBOOK_START = time.time()

# ================================ RUN CONFIG ================================
# Results go to /kaggle/working/results_<RUN_NAME>/ (+ results_<RUN_NAME>.zip).
RUN_NAME = 'dev60_baseline'

# Fixed 60-task dev set, chosen deterministically from tasks.jsonl: stratified by
# repo in proportion to the public set (fastapi 31 / rich 22 / requests 6 /
# httpx 1), and within each repo evenly spaced by created_at so old and new code
# are both covered. Ordered so every prefix stays repo-balanced: if the run is
# cut short, the finished part is still a fair sample. Keep this list fixed
# across runs so results compare task-by-task.
DEV_IDS = [
    'fastapi_13713', 'rich_3063', 'requests_6629', 'httpx_3672', 'fastapi_11194', 'rich_3043',
    'fastapi_12942', 'rich_3067', 'fastapi_14246', 'fastapi_14262', 'rich_3105', 'fastapi_14349',
    'requests_6589', 'rich_2725', 'fastapi_5624', 'fastapi_14361', 'rich_3278', 'fastapi_14303',
    'rich_3468', 'fastapi_14297', 'rich_3470', 'fastapi_13207', 'requests_7205', 'fastapi_14430',
    'rich_3454', 'fastapi_14448', 'rich_3480', 'fastapi_14455', 'fastapi_14458', 'rich_3506',
    'fastapi_14482', 'rich_3535', 'requests_7328', 'fastapi_14487', 'rich_3675', 'fastapi_14371',
    'fastapi_14605', 'rich_3777', 'fastapi_14479', 'rich_3718', 'fastapi_14786', 'requests_7427',
    'fastapi_14794', 'rich_3934', 'fastapi_14616', 'rich_3894', 'fastapi_14873', 'rich_3938',
    'fastapi_14953', 'fastapi_14978', 'rich_3944', 'requests_7502', 'fastapi_15023', 'rich_4006',
    'fastapi_15280', 'fastapi_15589', 'rich_4075', 'fastapi_15745', 'rich_4077', 'fastapi_15785',
]

# Stop STARTING new tasks once one more might not finish before this many hours
# (since notebook start). Kaggle kills sessions at 12 h; leave room for the
# packaging cell. Every finished task is already on disk either way.
MAX_WALL_HOURS = 11.0

# Your submission: path to submission.zip or to a directory containing
# agent.yaml. None = auto-detect among the attached datasets (set it explicitly
# if more than one submission is attached).
SUBMISSION_INPUT = None

assert len(DEV_IDS) == len(set(DEV_IDS)), 'duplicate ids in DEV_IDS'
print(f'{RUN_NAME}: {len(DEV_IDS)} tasks, stop starting new tasks after {MAX_WALL_HOURS} h')

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Your Submission

In [ ]:
import hashlib
import json
import zipfile

from adk_submission import validate_directory
from swegemma.config import build_submission_limits
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
AGENT_DIR = WORKING_DIR / 'submission_src'


def _zip_has_root_agent(z: Path) -> bool:
    try:
        with zipfile.ZipFile(z) as zf:
            return 'agent.yaml' in zf.namelist()
    except zipfile.BadZipFile:
        return False


def find_submission() -> Path:
    """Locate the attached submission: a zip with agent.yaml at its root, or a
    directory holding agent.yaml (Kaggle may auto-extract uploaded zips)."""
    if SUBMISSION_INPUT:
        roots = [Path(SUBMISSION_INPUT)]
    else:
        inp = Path('/kaggle/input')
        roots = [p for p in (inp / 'datasets').glob('*/*') if p.is_dir()]
        roots += [p for p in inp.iterdir() if p.is_dir() and p.name not in ('competitions', 'models', 'datasets')]
        roots = [r for r in roots if r.resolve() != WHEELHOUSE_DIR.resolve()]
    zips, dirs, bad = [], [], []
    for root in roots:
        if root.is_file():
            (zips if _zip_has_root_agent(root) else bad).append(root)
            continue
        for z in root.rglob('*.zip'):
            (zips if _zip_has_root_agent(z) else bad).append(z)
        dirs += [p.parent for p in root.rglob('agent.yaml')]
    if bad:
        print('Ignoring zips without agent.yaml at the archive root (the real submission would be rejected):', bad)
    candidates = zips or dirs
    if len(candidates) != 1:
        raise RuntimeError(
            f'Expected exactly one submission, found zips={zips} dirs={dirs}. '
            'Attach your submission dataset, or set SUBMISSION_INPUT in the run config.'
        )
    return candidates[0]


def submission_fingerprint(root: Path) -> str:
    """sha256 over (relative path, content) of every file. Must match scripts/package_submission.py."""
    h = hashlib.sha256()
    files = sorted((p for p in root.rglob('*') if p.is_file()), key=lambda p: p.relative_to(root).as_posix())
    for p in files:
        h.update(p.relative_to(root).as_posix().encode())
        h.update(b'\0')
        h.update(p.read_bytes())
        h.update(b'\0')
    return h.hexdigest()[:12]


SUBMISSION_SRC = find_submission()
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
if SUBMISSION_SRC.suffix == '.zip':
    with zipfile.ZipFile(SUBMISSION_SRC) as zf:
        zf.extractall(AGENT_DIR)
else:
    shutil.copytree(SUBMISSION_SRC, AGENT_DIR)

# Same structural checks the real submission goes through -- fail here, not after vLLM startup
for p in AGENT_DIR.rglob('*'):
    assert not p.is_symlink(), f'Symlinks are not permitted: {p.relative_to(AGENT_DIR)}'
limits, gen_constraints = build_submission_limits()
validate_directory(AGENT_DIR, limits=limits)

SUBMISSION_FINGERPRINT = submission_fingerprint(AGENT_DIR)
tasks = load_tasks(TASKS_PATH)

print(f'Submission source: {SUBMISSION_SRC}')
print(f'Submission fingerprint: {SUBMISSION_FINGERPRINT}  <- must match scripts/package_submission.py')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(AGENT_DIR).as_posix()}')
for name in ('agent.yaml', 'eval_config.yaml', 'configs/sampling.yaml'):
    if (AGENT_DIR / name).is_file():
        print(f'--- {name}\n' + (AGENT_DIR / name).read_text(encoding='utf-8').strip())
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')

### 2b. Environment Probe (diagnostic, safe to skip)

In [ ]:
# Diagnostic only: a few seconds, no GPU, safe to skip. Collects what is needed
# to fix the env_error bucket seen in the 1016 run (fastapi tests missing
# inline_snapshot / dirty_equals, requests tests with a broken httpbin fixture):
# are those packages in the offline wheels, and what does the harness's
# verify_task look like (to later verify reference patches in this env).
import importlib
import inspect

from swegemma.evaluate import Evaluator

env_probe = {}
_want = ('inline_snapshot', 'dirty_equals', 'pytest_httpbin', 'httpbin')
_wheels = sorted(w.name for w in (DATA_DIR / 'wheels').glob('*.whl'))
env_probe['wheels_total'] = len(_wheels)
env_probe['wheels_matching_missing_deps'] = [w for w in _wheels if any(k in w.lower().replace('-', '_') for k in _want)]
env_probe['importable_in_notebook'] = {m: importlib.util.find_spec(m) is not None for m in ('inline_snapshot', 'dirty_equals', 'pytest_httpbin')}
env_probe['verify_task'] = None
for _modname in ('swegemma.harness.verification', 'swegemma.verification', 'swegemma.evaluate', 'swegemma'):
    try:
        _m = importlib.import_module(_modname)
    except ImportError:
        continue
    if hasattr(_m, 'verify_task'):
        env_probe['verify_task'] = f'{_modname}.verify_task{inspect.signature(_m.verify_task)}'
        break
env_probe['evaluator_methods'] = [n for n in dir(Evaluator) if not n.startswith('__')]

(WORKING_DIR / 'env_probe.json').write_text(json.dumps(env_probe, indent=2))
print(json.dumps(env_probe, indent=2))

## 3. Start vLLM Server

In [ ]:
import re

import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct.
# A custom submission without an adapters/ directory simply runs with LoRA disabled.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4. Run the Dev Set (Phase 1 Inference + Phase 2 Verification)

In [ ]:
import asyncio
import concurrent.futures
import traceback

import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


def _maybe(obj, *names):
    """First non-None attribute among `names` -- harness result field names may vary."""
    for n in names:
        v = getattr(obj, n, None)
        if v is not None:
            return v
    return None


# Read optional evaluation settings from the submission's eval_config.yaml / eval_config.yml
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# --- dev set -------------------------------------------------------------------
_by_id = {t.instance_id: t for t in tasks}
_missing = [i for i in DEV_IDS if i not in _by_id]
assert not _missing, f'DEV_IDS not found in tasks.jsonl: {_missing}'
DEV_TASKS = [_by_id[i] for i in DEV_IDS]

# --- results layout read by eval/triage.py and eval/trace_stats.py --------------
# The Evaluator itself writes traces/ and logs/ here; this cell adds
# dev_results.jsonl, patches/, test_outputs/, submission.parquet, run_meta.json.
RESULTS_DIR = WORKING_DIR / f'results_{RUN_NAME}'
(RESULTS_DIR / 'patches').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'test_outputs').mkdir(parents=True, exist_ok=True)
DEV_JSONL = RESULTS_DIR / 'dev_results.jsonl'
done = set()
if DEV_JSONL.exists():  # re-running this cell resumes instead of starting over
    done = {json.loads(l)['id'] for l in DEV_JSONL.read_text(encoding='utf-8').splitlines() if l.strip()}
    print(f'Resuming: {len(done)}/{len(DEV_TASKS)} tasks already in {DEV_JSONL}')

_sampling_path = AGENT_DIR / 'configs' / 'sampling.yaml'
run_meta = {
    'run_name': RUN_NAME,
    'submission_source': str(SUBMISSION_SRC),
    'submission_fingerprint': SUBMISSION_FINGERPRINT,
    'model': declared_model,
    'eval_config': eval_section,
    'sampling': yaml.safe_load(_sampling_path.read_text(encoding='utf-8')) if _sampling_path.is_file() else None,
    'submission_files': {
        p.relative_to(AGENT_DIR).as_posix(): p.read_text(encoding='utf-8')
        for p in sorted(AGENT_DIR.rglob('*')) if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.py')
    },
    'dev_ids': DEV_IDS,
    'started_at_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
}
(RESULTS_DIR / 'run_meta.json').write_text(json.dumps(run_meta, indent=2, default=str), encoding='utf-8')

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
evaluator = Evaluator(eval_config)

# worst case for one more task: agent budget + Phase 2 verification + sandbox setup
_task_worst_h = (max_time_minutes + verification_timeout_seconds / 60 + 3) / 60

for idx, task in enumerate(DEV_TASKS, start=1):
    if task.instance_id in done:
        continue
    elapsed_h = (time.time() - NOTEBOOK_START) / 3600
    if elapsed_h + _task_worst_h > MAX_WALL_HOURS:
        print(f'Stopping at {elapsed_h:.2f} h: another task might not finish before MAX_WALL_HOURS={MAX_WALL_HOURS}.')
        break
    print(f'[{idx}/{len(DEV_TASKS)}] {task.instance_id} ({task.repo})  t={elapsed_h:.2f}h')

    t0 = time.time()
    result, crash = None, ''
    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(DEV_TASKS))
    except Exception as e:  # one broken task must not end a multi-hour run
        crash = f'{type(e).__name__}: {e}'
        traceback.print_exc()
    wall_seconds = time.time() - t0

    patch = str(_maybe(result, 'agent_patch', 'patch') or '').strip()
    has_patch = bool(patch) and patch != 'NO_PATCH'
    if has_patch:
        (RESULTS_DIR / 'patches' / f'{task.instance_id}.patch').write_text(patch, encoding='utf-8')

    # Phase 2 pytest log, if the result object carries it
    test_log = _maybe(result, 'test_output', 'test_log', 'verification_output', 'pytest_output')
    test_log_rel = None
    if test_log:
        test_log_rel = f'test_outputs/{task.instance_id}.log'
        (RESULTS_DIR / test_log_rel).write_text(str(test_log), encoding='utf-8')

    row = {
        'id': task.instance_id,
        'repo': task.repo,
        'resolved': bool(_maybe(result, 'resolved') or False),
        'test_exit_code': _maybe(result, 'test_exit_code', 'exit_code'),
        'patch_chars': len(patch) if has_patch else 0,
        'tool_calls': _maybe(result, 'tool_calls'),
        'duration_seconds': _maybe(result, 'duration_seconds', 'duration'),
        'wall_seconds': wall_seconds,  # setup + agent + verification
        'max_tool_calls': max_tool_calls,
        'max_time_minutes': max_time_minutes,
        'error': str(_maybe(result, 'error', 'agent_error') or crash or ''),
        # triage derives these from tasks.jsonl (--tasks) when the harness doesn't expose them
        'fail_to_pass': _maybe(task, 'fail_to_pass', 'FAIL_TO_PASS'),
        'pass_to_pass': _maybe(task, 'pass_to_pass', 'PASS_TO_PASS'),
        'test_log': test_log_rel,
    }
    with DEV_JSONL.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, default=str) + '\n')
    done.add(task.instance_id)
    print(
        f"  -> resolved={row['resolved']} patch_chars={row['patch_chars']} tool_calls={row['tool_calls']} "
        f"agent={row['duration_seconds'] or 0:.0f}s wall={wall_seconds:.0f}s"
        + (f" error={row['error'][:120]}" if row['error'] else '')
    )

# --- wrap up: submission.parquet + run summary (also correct after a resume) ---
rows = [json.loads(l) for l in DEV_JSONL.read_text(encoding='utf-8').splitlines() if l.strip()]
predictions = []
for r in rows:
    p = RESULTS_DIR / 'patches' / f"{r['id']}.patch"
    predictions.append({'id': r['id'], 'prediction': p.read_text(encoding='utf-8') if p.exists() else 'NO_PATCH'})
submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
submission_df.to_parquet(RESULTS_DIR / 'submission.parquet')

resolved = sum(r['resolved'] for r in rows)
run_meta.update(
    finished_at_utc=time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
    tasks_done=len(rows),
    resolved=resolved,
    notebook_hours=(time.time() - NOTEBOOK_START) / 3600,
)
(RESULTS_DIR / 'run_meta.json').write_text(json.dumps(run_meta, indent=2, default=str), encoding='utf-8')

print(f'\nDone {len(rows)}/{len(DEV_TASKS)} tasks, resolved {resolved} ({resolved / max(1, len(rows)):.1%})')
for repo in sorted({r['repo'] for r in rows}):
    rr = [r for r in rows if r['repo'] == repo]
    print(f"  {repo:<18} {sum(r['resolved'] for r in rr)}/{len(rr)}")

## 5. Package Results for Download

In [ ]:
# Bundle everything triage needs into one file for the Output tab
archive = Path(shutil.make_archive(str(WORKING_DIR / f'results_{RUN_NAME}'), 'zip', root_dir=RESULTS_DIR))
print(f'Created {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)')

_thinking = ((run_meta.get('sampling') or {}).get('thinking_config') or {}).get('thinking_budget', 2048)
print('\nDownload it from the Output tab, then in the repo:')
print(f'  mkdir -p exp_res/results_{RUN_NAME} && unzip results_{RUN_NAME}.zip -d exp_res/results_{RUN_NAME}')
print(f'  python eval/triage.py --results-dir exp_res/results_{RUN_NAME} --tasks tasks.jsonl')
print(f'  python eval/trace_stats.py --results-dir exp_res/results_{RUN_NAME} --thinking-budget {_thinking}')